# 01 — Analyse exploratoire

À exécuter après `python -m src.pipeline` (ou au moins l'extraction et la préparation).
Objectif : comprendre la demande horaire BIXI et la qualité des données avant de modéliser.

In [ ]:
import os, sys
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, os.getcwd())
import pandas as pd
import matplotlib.pyplot as plt
from src.common.config import charger_config
cfg = charger_config()

## 1. Données brutes : volumes, colonnes, valeurs manquantes

In [ ]:
from src.preprocessing.preparation import lire_deplacements, nettoyer
brut = lire_deplacements(cfg)
print(len(brut), "déplacements ;", brut["station"].nunique(), "stations")
print("Valeurs manquantes :\n", brut.isna().sum())
propre = nettoyer(brut, cfg)
print("Doublons exacts retirés et hors période :", len(brut) - len(propre))

## 2. Demande horaire préparée

In [ ]:
df = pd.read_parquet("data/processed/demande_horaire.parquet")
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["demande"].clip(upper=df["demande"].quantile(0.99)).hist(bins=40, ax=axes[0]); axes[0].set_title("Distribution de la demande (part de zéros : %.0f %%)" % (100 * (df["demande"] == 0).mean()))
df.assign(h=df["heure"].dt.hour, fds=df["heure"].dt.dayofweek >= 5).groupby(["h", "fds"])["demande"].mean().unstack().plot(ax=axes[1]); axes[1].set_title("Profil horaire : semaine (False) et fin de semaine (True)")
df.groupby("station")["demande"].mean().sort_values().plot.barh(ax=axes[2]); axes[2].set_title("Demande moyenne par station")
plt.tight_layout()

## 3. Effet de la météo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby(pd.cut(df["temperature"], 10), observed=True)["demande"].mean().plot.bar(ax=axes[0]); axes[0].set_title("Demande moyenne selon la température")
df.groupby(df["precipitation"] > 0)["demande"].mean().plot.bar(ax=axes[1]); axes[1].set_title("Demande moyenne : sans pluie / avec pluie")
plt.tight_layout()

## 4. Autocorrélation : justification des retards 1 h, 24 h et 168 h

In [ ]:
station = df["station"].iloc[0]
serie = df[df["station"] == station].set_index("heure")["demande"]
pd.Series({k: serie.autocorr(k) for k in [1, 2, 3, 12, 24, 48, 168]}).plot.bar(title=f"Autocorrélation — {station}")

## 5. Constats

*À rédiger avec vos propres résultats :* part de zéros, saisonnalités, effet météo, stations atypiques, limites des données.